# Proyecto 1 — Monitoreo transaccional

**Curso:** Deep Learning 2026  
**Estado:** etapas 0 y 1 — diseño previo a cualquier resultado  
**Integrantes:** _completar nombres y apellidos_

> Este notebook no genera datos ni entrena modelos. Primero dejamos congelado qué queremos comparar y qué puede ver cada modelo.

## 1. Pregunta y ruta de datos

La pregunta es: **¿el orden cronológico de las transacciones aporta información adicional para detectar fraude que no pueda capturarse únicamente mediante variables agregadas?**

Usaremos la **Ruta A: datos sintéticos con generador propio**. Esto nos permitirá crear mecanismos cuyo orden importe y otros que puedan detectarse con agregados. Así no diseñamos todo a favor de la GRU. El generador se implementará en la etapa 2; todavía no existe un dataset.

In [ ]:
from src.config import RANDOM_SEED

RANDOM_SEED

La semilla central será `42`. En las siguientes etapas se aplicará a Python `random`, NumPy, PyTorch, el generador y los cargadores. El generador deberá producir exactamente los mismos datos con la misma semilla, parámetros y versión del código.

## 2. Qué representa una predicción

Cada ejemplo corresponde a una **transacción objetivo de una tarjeta**. Simulamos la decisión que ocurre cuando esa operación intenta realizarse. Por eso el sistema puede ver el monto, canal, comercio, hora y demás campos presentes en la solicitud actual, además del historial estrictamente anterior de la tarjeta. No puede ver el resultado futuro de la operación, `is_fraud`, `fraud_type` ni transacciones posteriores.

Todos los modelos devolverán `risk_score ∈ [0, 1]`. No aplicamos todavía un threshold.

- **Modelo secuencial:** 12 eventos anteriores como candidato inicial y la operación objetivo como paso actual. Padding y máscara representarán historiales cortos.
- **Modelo agregado:** operación actual y estadísticas calculadas solo con el historial permitido.

Doce eventos es un comienzo manejable para representar ráfagas cortas, no una elección final. Solo TRAIN y VALIDATION podrán justificar cambiarlo.

## 3. Split temporal, TEST cerrado y leakage

La partición inicial será global y cronológica: primer 70 % para TRAIN, siguiente 15 % para VALIDATION y último 15 % para TEST. No habrá shuffle. Los bloques con el mismo timestamp no se partirán entre conjuntos y más adelante se comprobará:

```python
assert train.timestamp.max() < validation.timestamp.min()
assert validation.timestamp.max() < test.timestamp.min()
```

El historial causal de un objetivo puede cruzar hacia atrás una frontera, porque en producción ese pasado sí estaría disponible. La pertenencia al split la determina la transacción objetivo.

**TEST se usará una sola vez al final.** No servirá para variables, escaladores, encoders, longitud, arquitectura, hiperparámetros, elección de C, threshold ni para decidir si una hipótesis funciona.

Toda transformación aprendida hará `fit` exclusivamente con TRAIN y luego `transform` sobre cada split. Los agregados serán causales; no se usarán estadísticas calculadas al final de la vida de una tarjeta ni información futura.

## 4. Modelos que compararemos

### A — baseline sin orden

Candidato inicial: `HistGradientBoostingClassifier`. Recibirá monto y campos actuales, promedio/desviación/máximo histórico reciente, conteos y frecuencia, diversidad de comercios y canales, tiempo desde la operación anterior y razón entre monto actual y promedio. No recibirá posiciones, listas ordenadas ni transiciones completas. Será un baseline competitivo: usar resúmenes temporales causales no equivale a reconstruir el orden.

### B — GRU

`eventos ordenados → representación numérica → GRU → capa densa → sigmoid → risk_score`

La GRU es el punto de partida por su capacidad secuencial, número moderado de parámetros y claridad. Consideramos RNN simple, LSTM, CNN temporal y Transformer. No afirmamos que GRU sea siempre mejor.

### C — híbrido

`GRU(secuencia) + red densa(agregados) → concatenación → capas densas → risk_score`

**Hipótesis previa:** la secuencia puede representar transiciones y orden, mientras los agregados resumen el comportamiento reciente; combinarlos podría aportar información complementaria.

**Criterio previo:** C será candidato útil si en VALIDATION supera el AUC-PR de B y su costo económico, usando thresholds elegidos con la misma regla en VALIDATION, no es mayor. Si la diferencia es pequeña, un bootstrap agrupado por tarjeta deberá respaldar que no es solo ruido. No se definirá el veredicto mirando TEST.

## 5. Evaluación

La métrica principal será **AUC-PR**, porque el fraude estará desbalanceado. En el threshold elegido después con VALIDATION reportaremos precision, recall y F1. Accuracy será, como máximo, una referencia secundaria.

El costo será:

`economic_cost = false_negatives × Q4,200 + false_positives × Q180`

El threshold se minimizará con VALIDATION, se congelará y solo entonces se aplicará a TEST. Hoy no existe ningún threshold seleccionado. También reportaremos costo por transacción para poder comparar splits de distinto tamaño.

## 6. Pruebas que podrían refutar nuestra explicación

1. **Permutación controlada:** barajaremos los eventos dentro de cada secuencia conservando valores, longitud, agregados y etiqueta. Compararemos B con orden original frente a varias permutaciones reproducibles. Si no hay caída, no diremos que B utilizó el orden.
2. **Recorte de historia:** compararemos 12 eventos anteriores contra 3 sobre el mismo universo de ejemplos. Antes de ejecutarlo predecimos que los fraudes que dependen de una cadena de acciones serán más difíciles con poco contexto.

Ambas pruebas quedan declaradas antes del entrenamiento y se desarrollarán primero con TRAIN/VALIDATION.

## 7. Datos sintéticos que diseñaremos en la etapa 2

Campos iniciales: `timestamp`, `card_id`, `amount`, `merchant_category`, `channel`, `hour`, `day_of_week`, `time_since_previous` y `distance_from_previous`. `is_fraud` será la etiqueta y `fraud_type` se reservará para auditoría; nunca serán predictores.

- **Card testing → cashout:** varias operaciones pequeñas cercanas seguidas por una grande. Reordenar los mismos eventos no debería representar necesariamente el patrón.
- **Cambio anormal de canal/comportamiento:** por ejemplo, ONLINE desconocido → ATM en poco tiempo respecto a un perfil normalmente POS.
- **Anomalía de monto:** gasto extraordinario respecto a una historia estable; será detectable razonablemente mediante agregados.

También habrá vuelos, vacaciones, gastos extraordinarios legítimos, cambios válidos de canal, ráfagas de compras y usuarios irregulares. Esperamos que un viaje legítimo con canal nuevo, distancia alta y monto grande sea un caso difícil. Auditaremos que canal, periodo, comercio o rango de monto no delaten por sí solos la etiqueta.

## 8. Generación de datos sintéticos

Elegimos datos sintéticos para controlar qué señales existen y poder evaluar el valor del orden. Cada tarjeta representa un cliente con preferencias latentes de monto, frecuencia, horario, canal, categorías y distancia. Los cinco perfiles interpretables son regular, online, alto gasto, variable y viajero. Cada fila es una operación cruda; todavía no es una ventana ni una secuencia de modelado.

Dataset Version 1 contiene 95,767 transacciones de 2,800 tarjetas entre 2025-01-01 y 2025-06-29. Las columnas son `transaction_id`, `card_id`, `timestamp`, `amount`, `merchant_category`, `channel`, `distance_from_home_km`, `is_international`, `is_fraud`, `fraud_type`, `fraud_stage`, `customer_profile` y `hard_negative_type`. IDs, etiquetas, etapas, perfil y hard negatives son metadata, no features.

La tasa de fraude es 1.6446% (1,575 filas). Hay 38,100 operaciones POS, 30,868 ONLINE, 19,998 CONTACTLESS y 6,801 ATM. El monto medio es Q397.07 y la mediana Q158.53. La tarjeta mediana tiene 34 operaciones. Se marcaron 1,942 filas pertenecientes a hard negatives. Los gráficos descriptivos quedaron en `figures/`.

## 9. Mecanismos de fraude

| Mecanismo | Señal principal | ¿Depende del orden? |
|---|---|---|
| `testing_cashout` | pruebas pequeñas e intervalos cortos antes de una operación posterior | Alta |
| `channel_takeover` | transición anormal de canales y contexto personal | Media-alta |
| `amount_anomaly` | desviación del monto habitual | Baja |

Las pruebas y el cashout se etiquetan como fraude; `fraud_stage` distingue cada parte solo para análisis. Los tres mecanismos se distribuyen durante todo el periodo y no conocen el futuro split. Este diseño no afirma que una GRU vaya a ganar.

## 10. Casos legítimos difíciles

Incluimos viajes legítimos con canales y ubicaciones poco habituales, compras grandes válidas, shopping sprees y microcompras seguidas. Estos casos evitan que reglas simples como ‘monto alto’, ‘internacional’ o ‘muchas compras’ equivalgan a fraude. Esperamos equivocaciones cuando una compra grande válida ocurre durante un viaje o cuando un fraude imita cuidadosamente el comportamiento normal.

## 11. Reproducibilidad del dataset

- Semilla: `42`
- Versión del dataset: `1`
- Versión del generador: `1.0.0`
- Fingerprint SHA-256: `1f659a437a417e08b4274da79bfba8853887b2d4888d235c87e4a5d4ce5cf95d`

Dos generaciones pequeñas con seed 42 produjeron el mismo hash; seed 43 produjo uno distinto. Esta versión quedó congelada antes de entrenar: no cambiaremos los fraudes porque un resultado futuro no nos guste.

## 12. Justicia experimental y control de calidad

A, B y C usarán las mismas transacciones objetivo, etiqueta, horizonte, fronteras y TEST. Los historiales cortos no se descartarán solo para un modelo. B no tendrá futuro que A no tenga y C no recibirá columnas auxiliares.

- [x] No hay split aleatorio ni preprocessing ajustado con todos los datos.
- [x] TEST no se utilizó para tomar decisiones; solo se transformó con parámetros TRAIN.
- [x] No se eligió arquitectura con TEST ni threshold.
- [x] No se entrenaron modelos ni se inventaron métricas o resultados.
- [x] No se afirmó que el orden aporta.
- [x] La hipótesis de C y las falsificaciones quedaron escritas previamente.
- [x] La comparación tendrá el mismo universo y horizonte.

## 13. Partición temporal

Como queremos simular producción, no mezclamos operaciones futuras con entrenamiento. La pertenencia se define por el timestamp del target y cada timestamp completo queda en un solo split.

| Split | Inicio | Fin | Targets | Fraudes | Tasa |
|---|---|---|---:|---:|---:|
| TRAIN | 2025-01-01 04:00:48 | 2025-05-07 06:01:10 | 64,236 | 1,110 | 1.7280% |
| VALIDATION | 2025-05-07 06:10:15 | 2025-06-03 02:43:40 | 14,365 | 228 | 1.5872% |
| TEST | 2025-06-03 02:55:10 | 2025-06-29 23:59:59 | 14,366 | 222 | 1.5453% |

Una operación de TEST puede usar historia anterior de TRAIN, VALIDATION o TEST: esa información ya habría ocurrido. Nunca usamos las etiquetas históricas como features.

## 14. Construcción de ejemplos

Cada target tiene al menos una operación previa. Excluimos solo la primera operación de cada tarjeta (2,800 filas), dejando 92,967 ejemplos comunes. En empates temporales usamos `transaction_id` como desempate determinista. La secuencia guarda hasta 12 eventos anteriores y la operación actual va separada.

| Modelo | Operación actual | Agregados históricos | Secuencia ordenada |
|---|---|---|---|
| A | Sí | Sí | No |
| B | Sí | No | Sí |
| C | Sí | Sí | Sí |

## 15. Protección contra fuga de información

Los agregados usan exclusivamente historia anterior de la misma tarjeta. Los scalers, medianas de imputación y vocabularios se ajustaron solo con TRAIN. `PAD=0`, `UNK=1` y las categorías reales empiezan en 2. El left padding está acompañado por una máscara, por lo que no representa una compra. Etiquetas, mecanismos, etapas, hard negatives, perfiles e IDs permanecen separados de los inputs.

El fingerprint fuente fue verificado y los doce controles internos, siete pruebas automatizadas y revisiones manuales de agregados/secuencias pasaron.

## 16. Análisis exploratorio

Los tres mecanismos aparecen en cada split. `amount_anomaly` es el grupo más pequeño: 54/15/13 casos en TRAIN/VALIDATION/TEST. El 93.98% de los ejemplos tiene al menos 3 eventos previos, 84.94% al menos 6 y 66.87% al menos 12. La historia total mediana es 17.

Los gráficos `eda_temporal_activity.png`, `eda_fraud_mechanisms.png`, `eda_amount_by_mechanism.png`, `eda_history_length.png`, `eda_channel_by_label.png` y `eda_temporal_split.png` están en `figures/`. Son descripciones de integridad, no resultados predictivos.

## 17. Modelo A — Baseline sin orden

Antes de probar una red recurrente necesitábamos saber qué se puede lograr sin leer el orden de las operaciones. Modelo A usa la operación actual y 23 resúmenes causales del comportamiento anterior. No recibe secuencias, lags por posición, IDs, etiquetas ni metadata del generador.

Comparamos una regresión logística de sanity check y cinco configuraciones moderadas de HistGradientBoosting. Los pesos 0.50879 para legítimas y 28.93514 para fraude se calcularon solo con TRAIN. No hicimos resampling. La métrica congelada es AUC-PR / Average Precision mediante `average_precision_score`.

## 18. Candidatos de Modelo A

| Modelo | Parámetros principales | Train AP | Validation AP | Comentario |
|---|---|---:|---:|---|
| Logistic sanity | C=1, balanced | 0.474050 | 0.475988 | Señal lineal básica |
| HGB 01 | lr=.05, iter=150, leaves=15, min_leaf=50, L2=1 | 0.950572 | 0.893808 | Más regular |
| **HGB 02** | lr=.08, iter=180, leaves=31, min_leaf=40, L2=1 | **0.999970** | **0.910021** | Seleccionado |
| HGB 03 | lr=.05, iter=220, leaves=31, min_leaf=80, L2=2 | 0.997164 | 0.907099 | Sin mejora |
| HGB 04 | lr=.10, iter=150, leaves=15, min_leaf=80, L2=2 | 0.986016 | 0.899848 | Sin mejora |
| HGB 05 | lr=.05, iter=180, leaves=63, min_leaf=80, L2=5 | 0.998454 | 0.910294 | +0.000273; empate práctico y más complejo |

## 19. Resultado e interpretación de A

El candidato elegido alcanzó AP 0.999970 en TRAIN y 0.910021 en VALIDATION. El gap de 0.089948 indica sobreajuste y no debe ocultarse. La configuración HGB 05 fue apenas superior en validación, pero la diferencia de 0.000273 quedó dentro de la tolerancia previa de 0.001; conservamos HGB 02 por ser más simple.

Este resultado representa lo que logramos con información actual y agregada, sin entregar el orden completo. No permite afirmar todavía que el orden aporte o no aporte. La importancia por permutación mide dependencia predictiva, no causalidad.

En VALIDATION, la mediana de risk score fue 0.9985 para `testing_cashout`, 0.9888 para `channel_takeover` y 0.0409 para `amount_anomaly`. El comportamiento contrario a la expectativa inicial de monto debe investigarse después sin cambiar los datos. Microcompras y viajes legítimos tuvieron colas de riesgo altas, aunque medianas bajas.

## 20. Estado al cerrar Modelo A

Modelo A quedó congelado como baseline sin orden. TEST permanece sellado: no se cargaron sus labels, no se generaron predicciones y no existen métricas de TEST. Tampoco se eligió threshold, se calculó costo económico o se entrenaron B y C. El siguiente paso será entrenar B sobre las secuencias ordenadas con exactamente los mismos ejemplos y splits.

## 21. Modelo B — Secuencia ordenada

El segundo modelo recibe las operaciones anteriores en el orden en que ocurrieron. Elegimos una GRU porque mantiene un estado del historial con menos complejidad que una LSTM o Transformer y es suficiente para secuencias cortas. Es el equilibrio que decidimos probar, no una afirmación de superioridad universal.

```text
HISTORIAL → numéricas + embeddings → GRU → representación histórica ─┐
                                                                    ├→ Dense/ReLU → Dropout → logit → risk_score
OPERACIÓN ACTUAL → numéricas + embeddings → Dense/ReLU ─────────────┘
```

El historial tiene hasta 12 eventos previos, nunca el target. Comercio usa embedding 6 y canal embedding 3. La GRU unidireccional ignora left padding mediante packing. B no recibe agregados de A. Durante entrenamiento devuelve logits y usa `BCEWithLogitsLoss`; sigmoid se aplica solo para obtener risk scores.

## 22. Candidatos de Modelo B

| Candidato | Hidden | Capas | Dense | Dropout | LR | Parámetros | Best epoch | Train AP | Validation AP |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| B1 | 32 | 1 | 32 | 0.2 | .0010 | 7,675 | 9 | 0.823285 | 0.718066 |
| B2 | 64 | 1 | 32 | 0.2 | .0010 | 19,739 | 12 | 0.876947 | 0.672917 |
| **B3** | **64** | **1** | **64** | **0.4** | **.0008** | **25,499** | **9** | **0.836821** | **0.720674** |

Seleccionamos B3 por el mayor AP de VALIDATION. Early stopping usó esa misma métrica y paciencia 5 exclusivamente sobre VALIDATION. El `pos_weight=56.87027` proviene solo de TRAIN.

## 23. Resultado e interpretación de B

B3 obtuvo AP 0.836821 en TRAIN y 0.720674 en VALIDATION; el gap 0.116147 señala sobreajuste. Modelo A había obtenido 0.910021 en la misma validación, una diferencia B−A de −0.189348. **Esta comparación por sí sola NO demuestra todavía que el orden aporte información.** Debemos ejecutar la permutación controlada y la historia recortada antes de interpretar el uso del orden.

En VALIDATION, las medianas de score fueron 0.9972 para `testing_cashout`, 0.9867 para `channel_takeover` y 0.1955 para `amount_anomaly`. Viajes y microcompras legítimas muestran colas de riesgo altas. Estos son resultados descriptivos, sin threshold.

## 24. Estado al cerrar Modelo B

Modelo B quedó congelado. TEST sigue sellado y no se ejecutaron permutación, historia recortada, Modelo C, threshold o evaluación económica. El siguiente paso son las dos pruebas de falsificación predefinidas.

## 25. ¿El modelo realmente utilizó el orden?

Que B supere o no a A no responde por sí solo si la GRU utilizó el orden. Para aislarlo debemos cambiar el orden sin cambiar los eventos. La hipótesis escrita antes de ejecutar fue: **si B utiliza información contenida en el orden, esperamos que su AP disminuya al barajar los eventos dentro de cada historial; si permanece prácticamente igual, no tendremos evidencia suficiente.**

### Permutación controlada

Tomamos los mismos eventos de VALIDATION y los movimos como vectores completos. Mantuvimos PAD en su lugar, longitudes, current features, targets y checkpoint. `time_since_previous` permaneció pegado al evento para cambiar una sola cosa: la posición.

Con seeds 100–109, AP pasó de 0.720674 a una media de 0.284325 (std 0.020168; rango 0.255316–0.326197). La caída absoluta fue 0.436349 y la relativa 60.55%. La caída grande y consistente aporta evidencia fuerte de que B utilizaba información relacionada con el orden. No implica causalidad y B sigue debajo de A.

### Historia recortada

La hipótesis previa fue: **si algunos fraudes necesitan una cadena larga, limitar el historial a tres operaciones debería perjudicar su detección; si el desempeño permanece igual, el contexto largo probablemente aporta menos de lo supuesto.**

Conservamos los tres eventos reales más recientes y rellenamos las posiciones anteriores con PAD, sin reentrenar. AP cambió de 0.720674 a 0.710823: caída absoluta 0.009850 y relativa 1.37%. Todos los ejemplos de VALIDATION tenían más de tres eventos disponibles. La evidencia de que B necesite más de tres operaciones es limitada.

### Qué podemos afirmar hasta ahora

| Modelo o condición | Validation AP | Diferencia vs B original |
|---|---:|---:|
| A — agregados | 0.910021 | +0.189348 |
| B — secuencia original | 0.720674 | 0 |
| B — orden permutado (media) | 0.284325 | −0.436349 |
| B — historia máxima 3 | 0.710823 | −0.009850 |

Los resultados sugieren que B sí aprovecha el orden, pero esa información no fue suficiente para superar al baseline agregado. Tampoco encontramos evidencia fuerte de que las operaciones cuarta a duodécima sean necesarias para este checkpoint. TEST continúa sellado.

## 26. Apuesta C — Modelo híbrido

**Hipótesis previa:** creemos que combinar información secuencial con variables agregadas mejorará la detección porque cada representación resume un aspecto diferente. El criterio ya fijado dice que C debe superar el AP de B en VALIDATION y, posteriormente, no aumentar el costo económico.

La idea es directa: B conoce la secuencia; C recibe esa misma secuencia y operación actual, pero añade por otra rama los 23 resúmenes históricos de A. Se entrenó desde cero para no introducir pretraining como otra variable experimental.

```text
HISTORIA → embeddings + numéricas → GRU(64) ──────────────┐
CURRENT → embeddings + numéricas → Dense(64) ─────────────┼→ concat → Dense(64) → dropout → logit
AGREGADOS HISTÓRICOS → Dense(32) → ReLU → dropout ────────┘
```

### Resultado de C

| Candidato | Aggregate hidden | Fusion hidden | Dropout | LR | Parámetros | Best epoch | Train AP | Validation AP |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| C1 | 16 | 32 | 0.4 | .0008 | 22,235 | 12 | 0.880127 | 0.794085 |
| **C2** | **32** | **64** | **0.4** | **.0008** | **28,315** | **11** | **0.881519** | **0.815180** |
| C3 | 32 | 32 | 0.5 | .0008 | 23,131 | 12 | 0.882326 | 0.800241 |

C2 superó a B en 0.094507 AP (+13.11%) y tuvo un gap de 0.066339. Cumple la parte predictiva del criterio, pero la parte económica sigue pendiente. A, con 0.910021, continúa por encima de C.

### Ablación de agregados y veredicto

Con el mismo checkpoint sustituimos los 23 agregados escalados por cero, que representa el centro aprendido en TRAIN. AP cayó de 0.815180 a 0.429477, una diferencia de 0.385703. C sí está utilizando la rama agregada.

La apuesta se cumple parcialmente: existe apoyo predictivo en VALIDATION respecto a B, pero todavía no sabemos si cumple la condición económica. No hemos abierto TEST ni elegido threshold.

## 27. Comparación completa hasta ahora

| Condición | AP | Información |
|---|---:|---|
| A | 0.910021 | current + agregados |
| B original | 0.720674 | current + secuencia |
| B permutado | 0.284325 | current + mismos eventos sin orden |
| B historia ≤3 | 0.710823 | current + secuencia corta |
| C | 0.815180 | current + secuencia + agregados |

La permutación respalda que B usa el orden; el recorte indica poco valor adicional más allá de tres eventos; y C frente a B respalda complementariedad predictiva de los agregados. Nada de esto sustituye la futura evaluación económica o la única apertura de TEST.